[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Changing a Schema


## What you will be able to do

Change a table that already holds data. Add, rename and drop columns with `ALTER TABLE`, and know the
changes it refuses. Make any other change, such as a new constraint or a STRICT table, with the
twelve-step rebuild SQLite documents, without losing a row, an index, a view or a foreign key that
points at the table. Record the changes a database file has had in `PRAGMA user_version`, and write a
migration function that brings any copy of the file up to date and leaves a copy that already is
untouched.


## The idea

### The problem

The stations database has been in use for a year. A loader writes to it every hour, reports read its
`daily_means` view, and copies of the file sit on a laptop and on a server. Now it has to change.
Readings should refuse a second reading for the same hour and a temperature no thermometer could
show, and the table should be STRICT, which are rules the **Constraints** notebook declared in
`CREATE TABLE` and that the **Type Affinity** notebook said an existing table cannot be given in
place. Stations need a short code of their own.

Creating the tables again from nothing would throw the year away. SQLite's `ALTER TABLE` can add a
column, but it has no `ALTER COLUMN` and no `ADD CONSTRAINT`, so almost any other change means
building a new table and moving the rows into it, and every step of that can lose something without
an error. The indexes go when the old table is dropped. A foreign key follows a rename to a table
that is about to disappear. With foreign keys on, dropping a table deletes the rows that point at
it.

Then there are the copies. The laptop's copy has had one of the changes and the server's has had
none, so a program that changes a schema has to find out what a file already has before it changes
anything, and a change that fails halfway must leave the file as it was.

### What a schema change is

> A database's **schema** is its tables, columns, constraints, indexes, views and triggers, which
> SQLite keeps as `CREATE` statements in the table `sqlite_schema`. A **schema change**, or
> **migration**, alters the schema of a database that already holds data and keeps the data.
> SQLite's **`ALTER TABLE`** makes four changes in place: it renames a table, renames a column, adds
> a column and drops a column. Any other change takes a **table rebuild**: create the table as it
> should be under a new name, copy the rows into it, drop the old table and give the new one its
> name, in the order SQLite's documentation sets out in twelve steps. **`PRAGMA user_version`**
> reads and writes a number in the database file's header that SQLite leaves to the application, 0
> in a new file, which a program can use to count the migrations a file has had.

### Why it works that way

- **The schema is text.** `sqlite_schema` holds the `CREATE` statement of every table, index, view
  and trigger, and `ALTER TABLE` edits that text. Adding a column rewrites no row: a row written
  before the column existed has no value for it, and SQLite supplies the column's default when it
  reads the row, so the default has to be a constant, and a `NOT NULL` column needs one.
- **A rename carries through.** Since SQLite 3.25.0 and 3.26.0, renaming a table or a column also
  rewrites every index, view, trigger and foreign key that uses the name, which keeps a rename safe
  on its own, and decides the order of a rebuild.
- **A rebuild copies before it drops, and renames last.** The new table is built under a temporary
  name and takes the old name only after the old table is gone, so everything that named the old
  table names the new one without being touched. Renaming the old table first would carry every
  foreign key that points at it along to a table about to be dropped.
- **Foreign keys go off before the transaction begins.** With foreign keys on, `DROP TABLE` deletes
  every row first, so a table whose rows point at it with `ON DELETE CASCADE` would be emptied.
  `PRAGMA foreign_keys` is ignored inside a transaction, as the **Constraints** notebook showed, so
  it is switched off before `BEGIN`, and `PRAGMA foreign_key_check` looks for broken rows before
  `COMMIT`.
- **Schema changes are part of the transaction.** In SQLite, a `ROLLBACK` undoes `CREATE`, `DROP`
  and `ALTER` as it undoes any other change, so a rebuild that fails halfway leaves the old table
  exactly as it was. Not every database can promise that: MySQL commits before a statement that
  changes the schema.
- **`user_version` belongs to the application.** SQLite never reads the number, so it travels with
  every copy of the file, and a migration that sets it in its own transaction records exactly the
  changes the file has had.

### Where this shows up

Every application whose database outlives a release changes its schema, and a migration tool runs
these steps for it. The **SQLAlchemy, Deep Dive** guide's Alembic keeps a chain of migrations,
records the latest one a database has had in a table of its own, and rebuilds a SQLite table this
way in what it calls batch mode. Django, in the **Django, Deep Dive** guide, rebuilds a SQLite table
the same way when a model changes. PostgreSQL, in the **asyncpg and psycopg3, Deep Dive** guide,
changes a column's type or adds a constraint with `ALTER TABLE` alone, so the rebuild belongs to
SQLite. Android's `SQLiteOpenHelper` reads `user_version` to decide which of an app's upgrade steps
a database still needs. The **Backup and Copying** notebook shows how to take a safe copy of a
database, which is the cheapest protection before any change.

### What this notebook covers

- What a schema is, and where SQLite keeps it
- `ALTER TABLE ADD COLUMN`, and what the rows already there read back
- `RENAME COLUMN`, `RENAME TO` and `DROP COLUMN`, and what follows a rename
- The changes `ALTER TABLE` refuses
- The twelve-step rebuild, making `readings` STRICT with `UNIQUE` and `CHECK`
- Rebuilding a table other tables point at, with foreign keys off and `PRAGMA foreign_key_check`
- `PRAGMA user_version`, the number in the file's header
- Numbered migrations, and a function that runs only the ones a file has not had
- When to use `ALTER TABLE`, a unique index or a rebuild
- Two copies of one database brought up to date, one of them after a migration that failed
- Seven errors: a `NOT NULL` column with no default, `ADD CONSTRAINT`, a view that stops the rename,
  foreign keys switched off inside the transaction, the old table renamed first, a rebuild that did
  not make its indexes again, and a placeholder in a `PRAGMA`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:", autocommit=True)
conn.execute("CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL)")
conn.execute("INSERT INTO stations (name) VALUES ('Bergen'), ('Oslo')")
rows = "SELECT * FROM stations"
print(conn.execute("PRAGMA user_version").fetchone()[0], conn.execute(rows).fetchall())

conn.execute("BEGIN")
conn.execute("ALTER TABLE stations ADD COLUMN country TEXT NOT NULL DEFAULT 'Norway'")
conn.execute("PRAGMA user_version = 1")
conn.execute("COMMIT")
print(conn.execute("PRAGMA user_version").fetchone()[0], conn.execute(rows).fetchall())
conn.close()
```

```
0 [(1, 'Bergen'), (2, 'Oslo')]
1 [(1, 'Bergen', 'Norway'), (2, 'Oslo', 'Norway')]
```

The two rows were written before `country` existed, and read back with its default. The file's
`user_version` now says it has had one change, and the change and the number went into one
transaction, so neither can be saved without the other.


## Setup

Six imports, and the stations database as it first shipped: the two tables the **Tables and
Queries** notebook designed, with the year of readings, an index and a view. Every example changes a
copy of it, so the examples do not depend on one another, and a copy that goes wrong can simply be
made again.

- `sqlite3` builds the database and changes its schema
- `contextmanager`, from `contextlib`, turns the transaction around a schema change into a `with`
  block, in the worked examples
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the databases in it
- `shutil` copies the database, and removes the scratch folder at the end

`copy_of_shipped` makes a fresh copy of the shipped database. `open_database` opens a connection
that enforces foreign keys, with `autocommit=True`, so that every transaction in this notebook is
written in SQL, `BEGIN` and `COMMIT` included, and nothing ends one early: under the default
control, the **Transactions** notebook saw `executescript` commit an open transaction before it ran,
and under `autocommit=True` it commits nothing. `create_statement` reads the `CREATE` statement
SQLite keeps for a table, an index or a view.


In [1]:
import math
import shutil
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
SHIPPED = SCRATCH / "shipped.db"
SHIPPED.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(SHIPPED)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE INDEX readings_by_station_hour ON readings (station_id, hour);
    CREATE VIEW daily_means AS
        SELECT station_id, date(hour) AS day, ROUND(AVG(celsius), 1) AS mean
        FROM readings
        GROUP BY station_id, date(hour);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.executescript("""
    CREATE TABLE corrections (reading_id INTEGER NOT NULL, was REAL, now REAL);
    CREATE TRIGGER readings_after_correction AFTER UPDATE OF celsius ON readings BEGIN
        INSERT INTO corrections (reading_id, was, now) VALUES (old.id, old.celsius, new.celsius);
    END;
""")                                                  # created after the load, so it logs corrections only
build.commit()
build.close()


def copy_of_shipped(name):
    """A fresh copy of the database as it shipped, at scratch/<name>.db."""
    path = SCRATCH / f"{name}.db"
    shutil.copy(SHIPPED, path)
    return path


def open_database(path):
    """A connection that enforces foreign keys, with autocommit=True, so that the SQL writes its own transactions."""
    conn = sqlite3.connect(path, autocommit=True)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def create_statement(conn, name):
    """The CREATE statement SQLite keeps for a table, an index, a view or a trigger."""
    return conn.execute("SELECT sql FROM sqlite_schema WHERE name = ?", (name,)).fetchone()[0]


print("built", SHIPPED)


built scratch/shipped.db


## Worked examples

### What a schema is

`sqlite_schema` has a row for every table, index, view and trigger in the database, with its type,
its name, the table it belongs to and the `CREATE` statement that made it:


In [2]:
conn = open_database(copy_of_shipped("schema"))
for row in conn.execute("SELECT type, name, tbl_name FROM sqlite_schema ORDER BY rowid"):
    print(row)

print()
print(create_statement(conn, "readings"))
conn.close()


('table', 'stations', 'stations')
('table', 'readings', 'readings')
('index', 'readings_by_station_hour', 'readings')
('view', 'daily_means', 'daily_means')
('table', 'corrections', 'corrections')
('trigger', 'readings_after_correction', 'readings')

CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL)


Two tables, the index on `readings`, and the view, whose `tbl_name` is its own name. The `CREATE`
statement is kept as it was written, line break and spaces included, and SQLite reads it again every
time it opens the file. That text is the schema, and every change in this notebook is a change to it.

### Adding a column

`ADD COLUMN` adds a column at the end of a table. Here `stations` gains `code`, with no default, and
`country`, which is `NOT NULL` and so needs one:


In [3]:
conn = open_database(copy_of_shipped("add_column"))
conn.execute("ALTER TABLE stations ADD COLUMN code TEXT")
conn.execute("ALTER TABLE stations ADD COLUMN country TEXT NOT NULL DEFAULT 'NO'")

print(create_statement(conn, "stations"))
for row in conn.execute("SELECT * FROM stations ORDER BY id"):
    print(row)


CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL, code TEXT, country TEXT NOT NULL DEFAULT 'NO')
(1, 'Bergen', 60.39, None, 'NO')
(2, 'Oslo', 59.91, None, 'NO')
(3, 'Svalbard', 78.22, None, 'NO')
(4, 'Tromso', 69.65, None, 'NO')
(5, 'Kirkenes', 69.73, None, 'NO')


SQLite added both columns to the end of the stored `CREATE` statement and changed no row. The rows
were written before the columns existed, so they hold no values for them, and SQLite supplies each
column's default as it reads a row: `NULL` for `code`, and `'NO'` for `country`. That is why
`ADD COLUMN` takes the same moment on a table of five rows as on one of 35,040, and why it accepts
only a constant default, one value that suits every row it will ever be read into.

A column cannot be added with `UNIQUE`, but a unique index enforces the same rule and can be created
on a table that already has rows. Every station gets its code, and the index then refuses a second
station with the same one:


In [4]:
CODES = {"Bergen": "BGO", "Oslo": "OSL", "Svalbard": "LYR", "Tromso": "TOS", "Kirkenes": "KKN"}
conn.executemany("UPDATE stations SET code = ? WHERE name = ?", [(code, name) for name, code in CODES.items()])
conn.execute("CREATE UNIQUE INDEX stations_by_code ON stations (code)")
print(conn.execute("SELECT name, code FROM stations ORDER BY id").fetchall())

try:
    conn.execute("UPDATE stations SET code = 'OSL' WHERE name = 'Bergen'")
except sqlite3.IntegrityError as error:
    print("refused:", error)
conn.close()


[('Bergen', 'BGO'), ('Oslo', 'OSL'), ('Svalbard', 'LYR'), ('Tromso', 'TOS'), ('Kirkenes', 'KKN')]
refused: UNIQUE constraint failed: stations.code


### Renaming and dropping

`RENAME COLUMN` gives a column a new name, `RENAME TO` gives a table one, and `DROP COLUMN` removes a
column and its values. After every change, the statements that used the old name:


In [5]:
conn = open_database(copy_of_shipped("rename"))
conn.execute("ALTER TABLE readings RENAME COLUMN celsius TO temperature")
print(create_statement(conn, "daily_means"))

conn.execute("ALTER TABLE stations RENAME TO sites")
print()
print(create_statement(conn, "readings"))

conn.execute("ALTER TABLE sites DROP COLUMN latitude")
print()
print(create_statement(conn, "sites"))
conn.close()


CREATE VIEW daily_means AS
        SELECT station_id, date(hour) AS day, ROUND(AVG(temperature), 1) AS mean
        FROM readings
        GROUP BY station_id, date(hour)

CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES "sites" (id),
                           hour TEXT NOT NULL, temperature REAL)

CREATE TABLE "sites" (id INTEGER PRIMARY KEY, name TEXT NOT NULL)


The view now averages `temperature`, and `readings` now points at `"sites"`. Since SQLite 3.25.0 and
3.26.0, a rename reaches every index, view, trigger and foreign key that uses the name, so nothing is
left naming a column or a table that has gone. `DROP COLUMN`, from SQLite 3.35.0, rewrites every row
of the table without the column's values, and refuses a column that anything else in the schema
needs, as the next example shows.

### What ALTER TABLE refuses

Six changes `ALTER TABLE` will not make, with what it says about each:


In [6]:
conn = open_database(copy_of_shipped("refusals"))
for change in [
    "ALTER TABLE stations ADD COLUMN code TEXT UNIQUE",
    "ALTER TABLE stations ADD COLUMN station_number INTEGER PRIMARY KEY",
    "ALTER TABLE stations ADD COLUMN added TEXT DEFAULT CURRENT_TIMESTAMP",
    "ALTER TABLE stations ADD COLUMN elevation REAL DEFAULT 0 CHECK (elevation > 0)",
    "ALTER TABLE stations DROP COLUMN id",
    "ALTER TABLE readings DROP COLUMN celsius",
]:
    try:
        conn.execute(change)
        print("made:", change)
    except sqlite3.OperationalError as error:
        print(f"{error}\n    from {change}")
conn.close()


Cannot add a UNIQUE column
    from ALTER TABLE stations ADD COLUMN code TEXT UNIQUE
Cannot add a PRIMARY KEY column
    from ALTER TABLE stations ADD COLUMN station_number INTEGER PRIMARY KEY
Cannot add a column with non-constant default
    from ALTER TABLE stations ADD COLUMN added TEXT DEFAULT CURRENT_TIMESTAMP
CHECK constraint failed
    from ALTER TABLE stations ADD COLUMN elevation REAL DEFAULT 0 CHECK (elevation > 0)
cannot drop PRIMARY KEY column: "id"
    from ALTER TABLE stations DROP COLUMN id
error in view daily_means after drop column: no such column: celsius
    from ALTER TABLE readings DROP COLUMN celsius


SQLite's documentation lists the limits on an added column: it cannot be `UNIQUE` or a
`PRIMARY KEY`, cannot default to `CURRENT_TIME`, `CURRENT_DATE`, `CURRENT_TIMESTAMP` or an expression
in parentheses, needs a default other than `NULL` if it is `NOT NULL`, and has any `CHECK` tested
against the rows already there, where the default of 0 fails `elevation > 0`. A column cannot be
dropped while it is the primary key, or while an index covers it or a view reads it, as `celsius` is
read by `daily_means`.

And `ALTER TABLE` has no form at all for most other changes: a column's type, `NOT NULL` or a `CHECK`
on a column that already exists, a foreign key added or removed, a different primary key, or a table
made STRICT. Every one of those is a rebuild.

### The twelve steps

A rebuild makes a new table, as the old one should have been, and copies the rows into it. SQLite's
documentation gives the order in twelve steps:

1. If foreign keys are on, switch them off.
2. Begin a transaction.
3. Read the `CREATE` statements of the table's indexes, triggers and views from `sqlite_schema`.
4. Create the new table, as it should be, under a temporary name such as `readings_new`.
5. Copy the rows into it with `INSERT INTO ... SELECT`.
6. Drop the old table.
7. Rename the new table to the old name.
8. Create the indexes, triggers and views again.
9. Drop and create again any view the change affects.
10. Run `PRAGMA foreign_key_check`, and stop if it finds a broken row.
11. Commit.
12. Switch foreign keys back on.

`schema_change` below is steps 1, 2 and 10 to 12, a `with` block around a change that rolls the
transaction back if anything in the block raises. `rebuild` is steps 3 to 9. It drops the views
before it drops the table, not after: a view that names a table that is gone stops the rename in step
7, as a Common error shows. The table and view names it writes into its SQL come from this notebook's
code, never from input:


In [7]:
@contextmanager
def schema_change(conn):
    """Steps 1, 2 and 10 to 12: foreign keys off, one transaction, and a commit only if no foreign key is broken."""
    conn.execute("PRAGMA foreign_keys = OFF")                                                   # 1
    conn.execute("BEGIN")                                                                       # 2
    try:
        yield
        broken = conn.execute("PRAGMA foreign_key_check").fetchall()                            # 10
        if broken:
            raise sqlite3.IntegrityError(f"{len(broken)} rows break a foreign key, the first {broken[0]}")
        conn.execute("COMMIT")                                                                  # 11
    except BaseException:
        conn.execute("ROLLBACK")
        raise
    finally:
        conn.execute("PRAGMA foreign_keys = ON")                                                # 12


def rebuild(conn, table, create_new):
    """Steps 3 to 9: rebuild a table from create_new, which creates <table>_new, keeping its rows, indexes and views."""
    kept = [sql for (sql,) in conn.execute(                                                     # 3
        "SELECT sql FROM sqlite_schema WHERE tbl_name = ? AND type IN ('index', 'trigger') AND sql IS NOT NULL", (table,))]
    views = conn.execute("SELECT name, sql FROM sqlite_schema WHERE type = 'view'").fetchall()
    for name, _ in views:
        conn.execute(f'DROP VIEW "{name}"')                                                     # 9, before the drop
    conn.execute(create_new)                                                                    # 4
    old_columns = {column[1] for column in conn.execute(f'PRAGMA table_info("{table}")')}
    columns = ", ".join(f'"{column[1]}"' for column in conn.execute(f'PRAGMA table_info("{table}_new")')
                        if column[1] in old_columns)
    conn.execute(f'INSERT INTO "{table}_new" ({columns}) SELECT {columns} FROM "{table}"')      # 5
    conn.execute(f'DROP TABLE "{table}"')                                                       # 6
    conn.execute(f'ALTER TABLE "{table}_new" RENAME TO "{table}"')                              # 7
    for sql in kept + [sql for _, sql in views]:
        conn.execute(sql)                                                                       # 8 and 9


`readings` rebuilt as STRICT, with one reading for a station and hour and a `CHECK` on the
temperature, the change the **Type Affinity** notebook said an existing table needs a rebuild for:


In [8]:
NEW_READINGS = """
    CREATE TABLE readings_new (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        UNIQUE (station_id, hour)
    ) STRICT
"""

conn = open_database(copy_of_shipped("strict_readings"))
with schema_change(conn):
    rebuild(conn, "readings", NEW_READINGS)

print(create_statement(conn, "readings"))
print("STRICT:", conn.execute("SELECT strict FROM pragma_table_list WHERE name = 'readings'").fetchone()[0])
print("readings:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
print("schema:", conn.execute("SELECT type, name FROM sqlite_schema ORDER BY rowid").fetchall())
for hour, celsius in [("2025-01-01T00:00", 3.0), ("2026-01-01T00:00", 95.0)]:
    try:
        conn.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (1, ?, ?)", (hour, celsius))
    except sqlite3.IntegrityError as error:
        print("refused:", error)

conn.execute("UPDATE readings SET celsius = -3.9 WHERE id = 1")          # the trigger, after the rebuild
print("corrections logged:", conn.execute("SELECT * FROM corrections").fetchall())
conn.close()


CREATE TABLE "readings" (
        id         INTEGER PRIMARY KEY,
        station_id INTEGER NOT NULL REFERENCES stations (id),
        hour       TEXT NOT NULL,
        celsius    REAL CONSTRAINT plausible_celsius CHECK (celsius BETWEEN -90 AND 60),
        UNIQUE (station_id, hour)
    ) STRICT

STRICT: 1
readings: 35040
schema: [('table', 'stations'), ('table', 'corrections'), ('table', 'readings'), ('index', 'sqlite_autoindex_readings_1'), ('index', 'readings_by_station_hour'), ('trigger', 'readings_after_correction'), ('view', 'daily_means')]
refused: UNIQUE constraint failed: readings.station_id, readings.hour
refused: CHECK constraint failed: plausible_celsius
corrections logged: [(1, -3.6, -3.9)]


The rename gave the new table the old name, in quotes. All 35,040 readings made the move, the index,
the view and the trigger were made again, and `sqlite_autoindex_readings_1` is the index SQLite built
for `UNIQUE`, renamed along with its table. Both new rules now hold: a second reading for Bergen's
first hour is refused by `UNIQUE`, and 95 degrees by the `CHECK`, which the constraint's name says.
The trigger still logs a correction, so it survived the rebuild with its table. Step 5 is where the
new rules meet the old rows: had any stored reading broken either of them, the `INSERT ... SELECT`
would have failed and `schema_change` would have rolled the whole rebuild back, leaving `readings`
as it was.

### A table other tables point at

`readings` points at `stations`, so rebuilding `stations` drops a table that every reading refers to.
Here it becomes STRICT, with names of their own and latitudes on Earth:


In [9]:
NEW_STATIONS = """
    CREATE TABLE stations_new (
        id       INTEGER PRIMARY KEY,
        name     TEXT NOT NULL UNIQUE,
        latitude REAL NOT NULL CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90)
    ) STRICT
"""

conn = open_database(copy_of_shipped("strict_stations"))
with schema_change(conn):
    rebuild(conn, "stations", NEW_STATIONS)

print(create_statement(conn, "readings"))
joined = conn.execute("SELECT COUNT(*) FROM readings JOIN stations ON stations.id = readings.station_id").fetchone()[0]
print("readings joined to a station:", joined)
for change in ["INSERT INTO readings (station_id, hour, celsius) VALUES (99, '2026-01-01T00:00', 1.0)",
               "UPDATE stations SET latitude = 95 WHERE name = 'Oslo'"]:
    try:
        conn.execute(change)
    except sqlite3.IntegrityError as error:
        print("refused:", error)
conn.close()


CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL)
readings joined to a station: 35040
refused: FOREIGN KEY constraint failed
refused: CHECK constraint failed: plausible_latitude


`readings` still says `REFERENCES stations (id)`, as it was first written. The old `stations` was
dropped with foreign keys off, which deleted nothing, and the new table took the name in the last
rename, which no other table referred to, so nothing was rewritten. All 35,040 readings still join
to a station, foreign keys are enforced again, and the new `CHECK` refuses a latitude of 95.

With foreign keys on, step 6 would have gone differently. `DROP TABLE` deletes every row of the table
first, which the readings pointing at them would have refused. A child table declared
`ON DELETE CASCADE`, as the **Constraints** notebook's notes were, refuses nothing: it goes quietly.
On a copy whose readings cascade, with foreign keys on, dropping the stations empties them:


In [10]:
CASCADING_READINGS = NEW_READINGS.replace("REFERENCES stations (id)", "REFERENCES stations (id) ON DELETE CASCADE")

conn = open_database(copy_of_shipped("cascading"))
with schema_change(conn):
    rebuild(conn, "readings", CASCADING_READINGS)

print("readings before the drop:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
conn.execute("DROP TABLE stations")                      # foreign keys are on again by now
print("readings after it:       ", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
conn.close()


readings before the drop: 35040
readings after it:        0


Every reading gone, and nothing raised: `DROP TABLE` deleted the stations, and the cascade followed
the readings down. That is why step 1 of the twelve is `PRAGMA foreign_keys = OFF`, before the
transaction, and step 10 checks the whole database with `PRAGMA foreign_key_check` before committing.

### PRAGMA user_version

`PRAGMA user_version` reads the number, and `PRAGMA user_version = 1` writes it. Here it is set
inside a transaction that rolls back, set again for good, read from the file's own bytes, and read
again through a new connection:


In [11]:
path = copy_of_shipped("versions")
conn = open_database(path)
print("a new copy:", conn.execute("PRAGMA user_version").fetchone()[0])

conn.execute("BEGIN")
conn.execute("PRAGMA user_version = 1")
conn.execute("ROLLBACK")
print("after a rollback:", conn.execute("PRAGMA user_version").fetchone()[0])

conn.execute("PRAGMA user_version = 1")
conn.close()
with path.open("rb") as file:
    file.seek(60)
    print("bytes 60 to 63 of the file:", int.from_bytes(file.read(4), "big"))

conn = open_database(path)
print("a new connection:", conn.execute("PRAGMA user_version").fetchone()[0])
conn.close()


a new copy: 0
after a rollback: 0
bytes 60 to 63 of the file: 1
a new connection: 1


A new database starts at 0. The number is four bytes at offset 60 of the file's header, which SQLite
keeps for the application and never reads itself, so it goes wherever the file goes. Setting it is
part of the transaction it runs in, so the 1 that was rolled back was never saved. Its companion,
`PRAGMA application_id`, reads four bytes further on, at offset 68, for marking which program a file
belongs to.

### Numbered migrations

A migration is one numbered change, and a database's `user_version` is the number of the last one it
has had. The three changes so far become migrations 1 to 3, and `migrate` reads the version and runs
every later migration, each inside `schema_change` together with the `PRAGMA` that records its
number:


In [12]:
def make_readings_strict(conn):
    """Version 1: readings STRICT, one reading for a station and hour, and plausible temperatures."""
    rebuild(conn, "readings", NEW_READINGS)


def make_stations_strict(conn):
    """Version 2: stations STRICT, with names of their own and latitudes on Earth."""
    rebuild(conn, "stations", NEW_STATIONS)


def add_station_codes(conn):
    """Version 3: a code for every station, different for every station."""
    conn.execute("ALTER TABLE stations ADD COLUMN code TEXT")
    conn.executemany("UPDATE stations SET code = ? WHERE name = ?", [(code, name) for name, code in CODES.items()])
    conn.execute("CREATE UNIQUE INDEX stations_by_code ON stations (code)")


MIGRATIONS = [make_readings_strict, make_stations_strict, add_station_codes]


def migrate(conn, migrations=MIGRATIONS):
    """Run every migration the database has not had, each in its own transaction with its number, and return the numbers."""
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    applied = []
    for number in range(version + 1, len(migrations) + 1):
        with schema_change(conn):
            migrations[number - 1](conn)
            conn.execute(f"PRAGMA user_version = {number}")
        applied.append(number)
    return applied


conn = open_database(copy_of_shipped("migrated"))
print("applied:", migrate(conn))
print("applied on a second run:", migrate(conn))
print("version:", conn.execute("PRAGMA user_version").fetchone()[0])
strict_tables = conn.execute("SELECT name FROM pragma_table_list WHERE strict = 1 ORDER BY name").fetchall()
print("STRICT tables:", [name for (name,) in strict_tables])
conn.close()


applied: [1, 2, 3]
applied on a second run: []
version: 3
STRICT tables: ['readings', 'stations']


The first run applied all three, and the second found the database at version 3 and did nothing, so
`migrate` can run every time a program starts. The version is written into the SQL with an f-string,
since a `PRAGMA` takes no placeholder, and `number` is an integer this code counted. A new change
goes on the end of `MIGRATIONS`, and a migration is never edited once a copy of the file has had it,
since that copy will not run it again. A ladder of `if version < 1:` blocks does the same work, and
the list keeps every number in one place.

### ALTER TABLE, a unique index, or a rebuild

| Write | When | Why |
|---|---|---|
| `ALTER TABLE ... ADD COLUMN` | a new column with a constant default, or none | it edits only the stored `CREATE` statement, so it is as quick on a big table as on a small one |
| `CREATE UNIQUE INDEX` | a new rule that a column, or a set of columns, never repeats | it enforces what `UNIQUE` would on a table that already exists, and refuses to be created while a repeat is there |
| `ALTER TABLE ... RENAME` | a new name for a table or a column | the new name reaches every index, view, trigger and foreign key that used the old one |
| `ALTER TABLE ... DROP COLUMN` | a column nothing else in the schema uses | it rewrites the rows without the column, and refuses while an index, a view or a constraint needs it |
| the twelve-step rebuild | anything else: a type, `NOT NULL`, `CHECK`, a foreign key, a primary key, STRICT | it builds the table as it should be and copies the rows across in one transaction |

The default is `ALTER TABLE`, or a unique index, whenever one can make the change, and a rebuild when
neither can. Whichever it is, it goes into a numbered migration, so every copy of the file gets it
once.

### Two copies, brought up to date

The pieces of this notebook together, on two copies of the database at different versions. The
laptop's copy had the first migration last month. The server's copy has had none, and a loader that
retried has written one of Oslo's readings twice, which the `UNIQUE` in migration 1 will not accept:


In [13]:
laptop = open_database(copy_of_shipped("laptop"))
migrate(laptop, MIGRATIONS[:1])                                                   # last month, on the laptop
server = open_database(copy_of_shipped("server"))
server.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (2, '2025-06-01T12:00', 14.2)")   # the retry

for name, conn in [("laptop", laptop), ("server", server)]:
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    try:
        print(f"{name}: from version {version}, applied {migrate(conn)}")
    except sqlite3.IntegrityError as error:
        still = conn.execute("PRAGMA user_version").fetchone()[0]
        count = conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
        print(f"{name}: from version {version}, migration {still + 1} refused: {error}")
        print(f"{name}: still version {still}, with all {count:,} readings")

removed = server.execute("DELETE FROM readings WHERE id NOT IN (SELECT MIN(id) FROM readings GROUP BY station_id, hour)")
print(f"server: {removed.rowcount} duplicate removed, applied {migrate(server)}")

schema = "SELECT type, name, sql FROM sqlite_schema ORDER BY name"
print("the same schema on both:", laptop.execute(schema).fetchall() == server.execute(schema).fetchall())
laptop.close()
server.close()


laptop: from version 1, applied [2, 3]
server: from version 0, migration 1 refused: UNIQUE constraint failed: readings_new.station_id, readings_new.hour
server: still version 0, with all 35,041 readings
server: 1 duplicate removed, applied [1, 2, 3]
the same schema on both: True


The laptop's copy needed only migrations 2 and 3. The server's copy failed in migration 1, at the
copy in step 5, and the rollback left it at version 0 with every reading it had, the duplicate
included. With the duplicate gone, `migrate` ran all three, and the two copies now hold the same
schema, statement for statement.

### Where each part came from

| In the migration run | What it relies on | The section that showed it |
|---|---|---|
| `rebuild(conn, "readings", NEW_READINGS)` | the twelve steps, with the views dropped before the table | The twelve steps |
| `rebuild(conn, "stations", NEW_STATIONS)` | a rebuild of a table that other tables point at | A table other tables point at |
| `ADD COLUMN code TEXT` and `CREATE UNIQUE INDEX` | a column added in place, and a unique rule added after it | Adding a column |
| `with schema_change(conn)` | foreign keys off before `BEGIN`, and a check before `COMMIT` | The twelve steps |
| `PRAGMA user_version` read, and set with the change | a count of migrations, saved or rolled back with them | PRAGMA user_version |
| `migrate(conn)` | only the migrations a file has not had | Numbered migrations |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/13-changing-a-schema-solutions.ipynb).

**1.** On a copy of the shipped database, add a column to `stations`, declared as
`elevation REAL NOT NULL DEFAULT 0`, then print the table's `CREATE` statement and Tromso's row.


In [14]:
# your code here


**2.** On a copy, rename the table `readings` to `observations`, then print the `CREATE` statements
of `daily_means` and `readings_by_station_hour` to see what followed the rename.


In [15]:
# your code here


**3.** On a copy, try to rebuild `readings` with `celsius REAL NOT NULL`, using `schema_change` and
`rebuild`. Print the error, and show that the table still holds all its readings and is not STRICT.


In [16]:
# your code here


**4.** Write a version of `migrate` that refuses a database whose `user_version` is higher than the
number of migrations it knows, and try it on a copy whose version is 9.


In [17]:
# your code here


**5.** Write a fourth migration that adds `source TEXT NOT NULL DEFAULT 'station'` to `readings`, and
bring a copy of the shipped database up to date with all four.


In [18]:
# your code here


**6.** On a copy, with foreign keys off, insert a reading for a station 99 that does not exist, then
find it with `PRAGMA foreign_key_check` and print what the check reports.


In [19]:
# your code here


## Common errors

### sqlite3.OperationalError: Cannot add a NOT NULL column with default value NULL


In [20]:
conn = open_database(copy_of_shipped("not_null"))
conn.execute("ALTER TABLE stations ADD COLUMN country TEXT NOT NULL")


OperationalError: Cannot add a NOT NULL column with default value NULL

The rows already in `stations` have no value for the new column, and a column with no default gives
them `NULL`, which `NOT NULL` forbids. Give the column a default for those rows to read:


In [21]:
conn.execute("ALTER TABLE stations ADD COLUMN country TEXT NOT NULL DEFAULT 'NO'")
print(conn.execute("SELECT name, country FROM stations ORDER BY id").fetchall())
conn.close()


[('Bergen', 'NO'), ('Oslo', 'NO'), ('Svalbard', 'NO'), ('Tromso', 'NO'), ('Kirkenes', 'NO')]


### sqlite3.OperationalError: near "CONSTRAINT": syntax error


In [22]:
conn = open_database(copy_of_shipped("add_constraint"))
conn.execute("ALTER TABLE stations ADD CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90)")


OperationalError: near "CONSTRAINT": syntax error

`ADD CONSTRAINT` is how PostgreSQL, MySQL and SQL Server add a rule to a table, and SQLite's
`ALTER TABLE` has no such form, so it stopped at the word. `ALTER COLUMN`, for a new type or
`NOT NULL`, fails the same way, near `"ALTER"`. A `CHECK`, like any rule `ALTER TABLE` cannot add,
goes into a rebuild:


In [23]:
with schema_change(conn):
    rebuild(conn, "stations", NEW_STATIONS)

try:
    conn.execute("UPDATE stations SET latitude = 95 WHERE name = 'Oslo'")
except sqlite3.IntegrityError as error:
    print("refused:", error)
conn.close()


refused: CHECK constraint failed: plausible_latitude


### sqlite3.OperationalError: error in view daily_means: no such table: main.readings


In [24]:
conn = open_database(copy_of_shipped("view_in_the_way"))
with schema_change(conn):
    conn.execute(NEW_READINGS)
    conn.execute("""
        INSERT INTO readings_new (id, station_id, hour, celsius) SELECT id, station_id, hour, celsius FROM readings
    """)
    conn.execute("DROP TABLE readings")
    conn.execute("ALTER TABLE readings_new RENAME TO readings")


OperationalError: error in view daily_means: no such table: main.readings

The rebuild did steps 4 to 7 and left the view alone. A rename reads every view in the schema, to
carry the new name into it, and `daily_means` names `readings`, which step 6 had just dropped, so
the rename could not go ahead. `schema_change` rolled the transaction back, so nothing is lost:


In [25]:
print("readings:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
print("tables:", [name for (name,) in conn.execute("SELECT name FROM sqlite_schema WHERE type = 'table' ORDER BY name")])

with schema_change(conn):
    rebuild(conn, "readings", NEW_READINGS)
print("STRICT after rebuild:", conn.execute("SELECT strict FROM pragma_table_list WHERE name = 'readings'").fetchone()[0])
conn.close()


readings: 35040
tables: ['corrections', 'readings', 'stations']
STRICT after rebuild: 1


`rebuild` drops every view before it drops the table, and creates them again from their own `CREATE`
statements after the rename. A trigger on another table that names the rebuilt table needs the same
care, and this database has none.

### sqlite3.IntegrityError: FOREIGN KEY constraint failed


In [26]:
conn = open_database(copy_of_shipped("pragma_inside"))
conn.execute("BEGIN")
conn.execute("PRAGMA foreign_keys = OFF")
conn.execute(NEW_STATIONS)
conn.execute("INSERT INTO stations_new (id, name, latitude) SELECT id, name, latitude FROM stations")
conn.execute("DROP TABLE stations")


IntegrityError: FOREIGN KEY constraint failed

The `PRAGMA` came after `BEGIN`, and SQLite ignores `PRAGMA foreign_keys` inside a transaction, so
foreign keys were still on when `DROP TABLE` ran. With foreign keys on, `DROP TABLE` deletes every
row first, and the readings pointing at the stations refused. Had `readings` been declared
`ON DELETE CASCADE`, the drop would have gone through and taken every reading with it. The failed
statement leaves the transaction open, so roll it back, and switch foreign keys off before `BEGIN`,
as `schema_change` does:


In [27]:
print("foreign keys, inside that transaction:", conn.execute("PRAGMA foreign_keys").fetchone()[0])
conn.execute("ROLLBACK")

with schema_change(conn):
    rebuild(conn, "stations", NEW_STATIONS)
print("stations:", conn.execute("SELECT COUNT(*) FROM stations").fetchone()[0],
      "| readings:", conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
conn.close()


foreign keys, inside that transaction: 1
stations: 5 | readings: 35040


### sqlite3.OperationalError: no such table: main.stations_old


In [28]:
conn = open_database(copy_of_shipped("renamed_first"))
conn.execute("PRAGMA foreign_keys = OFF")
conn.execute("BEGIN")
conn.execute("ALTER TABLE stations RENAME TO stations_old")
conn.execute(NEW_STATIONS)
conn.execute("INSERT INTO stations_new (id, name, latitude) SELECT id, name, latitude FROM stations_old")
conn.execute("ALTER TABLE stations_new RENAME TO stations")
conn.execute("DROP TABLE stations_old")
conn.execute("COMMIT")
conn.execute("PRAGMA foreign_keys = ON")

conn.execute("INSERT INTO readings (station_id, hour, celsius) VALUES (1, '2026-01-01T00:00', 4.5)")


OperationalError: no such table: main.stations_old

The rebuild renamed the old table first, the order SQLite's documentation warns against, and the
rename did its job: it carried the new name into `readings`, whose foreign key then pointed at
`stations_old`, the table the rebuild went on to drop. Every new reading now looks for a table that
is gone:


In [29]:
print(create_statement(conn, "readings"))
broken = conn.execute("PRAGMA foreign_key_check").fetchall()
print(len(broken), "rows break a foreign key, the first", broken[0])
conn.close()


CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES "stations_old" (id),
                           hour TEXT NOT NULL, celsius REAL)
35040 rows break a foreign key, the first ('readings', 1, 'stations_old', 0)


Step 10 finds this before the commit. The same wrong order, inside `schema_change`, is rolled back
with every reading still pointing at `stations`, and the order in `rebuild`, create, copy, drop,
rename, never renames the table other tables point at:


In [30]:
conn = open_database(copy_of_shipped("renamed_first_checked"))
try:
    with schema_change(conn):
        conn.execute("ALTER TABLE stations RENAME TO stations_old")
        conn.execute(NEW_STATIONS)
        conn.execute("INSERT INTO stations_new (id, name, latitude) SELECT id, name, latitude FROM stations_old")
        conn.execute("ALTER TABLE stations_new RENAME TO stations")
        conn.execute("DROP TABLE stations_old")
except sqlite3.IntegrityError as error:
    print("rolled back:", error)
print(create_statement(conn, "readings"))
conn.close()


rolled back: 35040 rows break a foreign key, the first ('readings', 1, 'stations_old', 0)
CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL)


### No error, and two stations coded OSL: a rebuild that did not make its indexes again


In [31]:
conn = open_database(copy_of_shipped("lost_index"))
migrate(conn)
indexes = "SELECT name FROM sqlite_schema WHERE type = 'index' AND tbl_name = 'stations' ORDER BY name"
print("indexes on stations:", [name for (name,) in conn.execute(indexes)])

with schema_change(conn):                           # to make code NOT NULL
    conn.execute("""
        CREATE TABLE stations_new (
            id       INTEGER PRIMARY KEY,
            name     TEXT NOT NULL UNIQUE,
            latitude REAL NOT NULL CONSTRAINT plausible_latitude CHECK (latitude BETWEEN -90 AND 90),
            code     TEXT NOT NULL
        ) STRICT
    """)
    conn.execute("INSERT INTO stations_new (id, name, latitude, code) SELECT id, name, latitude, code FROM stations")
    conn.execute("DROP TABLE stations")
    conn.execute("ALTER TABLE stations_new RENAME TO stations")

print("indexes on stations:", [name for (name,) in conn.execute(indexes)])
conn.execute("UPDATE stations SET code = 'OSL' WHERE name = 'Bergen'")
print(conn.execute("SELECT name, code FROM stations WHERE code = 'OSL' ORDER BY name").fetchall())


indexes on stations: ['sqlite_autoindex_stations_1', 'stations_by_code']
indexes on stations: ['sqlite_autoindex_stations_1']
[('Bergen', 'OSL'), ('Oslo', 'OSL')]


The rebuild made `code` required and lost what kept it unique. `stations_by_code` belonged to the old
`stations`, and `DROP TABLE` drops a table's indexes and triggers with it, so the new table kept only
`sqlite_autoindex_stations_1`, the index for `UNIQUE` on `name`. Nothing failed, and Bergen and Oslo
now share a code. Step 3 reads the indexes before the drop, and step 8 makes them again, as `rebuild`
does. Here, put Bergen's code back and create the index again:


In [32]:
conn.execute("UPDATE stations SET code = 'BGO' WHERE name = 'Bergen'")
conn.execute("CREATE UNIQUE INDEX stations_by_code ON stations (code)")
try:
    conn.execute("UPDATE stations SET code = 'OSL' WHERE name = 'Bergen'")
except sqlite3.IntegrityError as error:
    print("refused:", error)
conn.close()


refused: UNIQUE constraint failed: stations.code


### sqlite3.OperationalError: near "?": syntax error


In [33]:
conn = open_database(copy_of_shipped("placeholder"))
conn.execute("PRAGMA user_version = ?", (3,))


OperationalError: near "?": syntax error

A placeholder can stand only where SQL takes a value, such as in `WHERE` or `VALUES`, and the number
in a `PRAGMA` is part of the statement's own syntax, so SQLite found a `?` where it expected a
number. Write the number into the SQL, through `int`, so that nothing but a number can end up there:


In [34]:
version = 3
conn.execute(f"PRAGMA user_version = {int(version)}")
print("version:", conn.execute("PRAGMA user_version").fetchone()[0])
conn.close()


version: 3


Last, every connection is closed, so this cell removes the scratch folder, with every copy of the
database in it:


In [35]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- SQLite keeps a schema as `CREATE` statements in `sqlite_schema`, and `ALTER TABLE` edits them: it
  renames a table or a column, adds a column and drops a column, and a rename reaches every index,
  view, trigger and foreign key that used the name.
- An added column changes no row, so a `NOT NULL` column needs a constant default, and no added
  column can be `UNIQUE` or a `PRIMARY KEY`. `CREATE UNIQUE INDEX` adds the uniqueness afterwards.
- Every other change is a rebuild: create the new table under a temporary name, copy the rows, drop
  the old table, rename the new one, and create the indexes, triggers and views again.
- Drop the views that name the table before dropping the table, and never rename the old table first.
- Switch foreign keys off before `BEGIN`, since the `PRAGMA` is ignored inside a transaction and
  `DROP TABLE` with foreign keys on deletes every row, and run `PRAGMA foreign_key_check` before
  `COMMIT`.
- `PRAGMA user_version` is a number in the file's header for the application. Set it in the same
  transaction as the change it counts, written into the SQL, since a `PRAGMA` takes no placeholder.
- A migration function reads the version and runs only the later migrations, and a migration that
  fails leaves the file at the version it had.


## What is next

The **executemany** notebook loads rows in bulk: `executemany` with one transaction around it, a
generator in place of a list, and what `lastrowid` does and does not report afterwards.


---

&#8592; **Previous:** [Constraints](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/12-constraints.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [executemany](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/14-executemany.ipynb) &#8594;
